In [8]:
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split

In [5]:
# 1. Cargar el dataset de la clase
df = pd.read_csv("1980sClassics.csv")

# 2. Transformar 'Duration' (formato 'mm:ss') a una variable numérica continua en segundos
df["duration_seconds"] = df["Duration"].apply(
    lambda x: int(x.split(":")[0]) * 60 + float(x.split(":")[1])
)

# 3. Opcional: Eliminar la columna original en formato texto
df = df.drop(columns=["Duration"])

# Ver las primeras filas
df[["Track", "Artist", "duration_seconds"]].head()

,Track,Artist,duration_seconds
0,Babe,Styx,218.0
1,The Rose,Bette Midler,244.0
2,Cars,Gary Numan,248.0
3,Magic,Olivia Newton-John,137.0
4,We Don’t Talk Anymore,Cliff Richard,217.0


In [6]:
# 1. Option A: Frequency Encoding
df["artist_freq"] = df["Artist"].map(
    df["Artist"].value_counts(normalize=True)
)

# 2. Option B: Target Encoding (asumiendo 'Popularity' como target)
target_encoder = TargetEncoder()
df["artist_target_enc"] = target_encoder.fit_transform(
    df["Artist"], df["Popularity"]
)

df[["Artist", "artist_freq", "artist_target_enc"]].head()

,Artist,artist_freq,artist_target_enc
0,Styx,0.005010,57.826767
1,Bette Midler,0.002004,60.262267
2,Gary Numan,0.001002,60.899465
3,Olivia Newton-John,0.006012,59.047065
4,Cliff Richard,0.003006,58.709901


In [7]:
# 1. Discretización por igual amplitud (cut)
df["speech_width"] = pd.cut(df["Speechiness"], bins=4)

# 2. Discretización por cuantiles / frecuencia (qcut)
df["speech_quantile"] = pd.qcut(df["Speechiness"], q=4)

# Comparación de frecuencias resultantes
print("--- Igual Amplitud (cut) ---")
print(df["speech_width"].value_counts().sort_index())

print("\n--- Cuantiles (qcut) ---")
print(df["speech_quantile"].value_counts().sort_index())

--- Igual Amplitud (cut) ---
speech_width
(0.0222, 0.148]    946
(0.148, 0.273]      33
(0.273, 0.399]      16
(0.399, 0.524]       3
Name: count, dtype: int64

--- Cuantiles (qcut) ---
speech_quantile
(0.0217, 0.0317]    252
(0.0317, 0.0393]    247
(0.0393, 0.0564]    249
(0.0564, 0.524]     250
Name: count, dtype: int64


In [9]:
# 1. Verificar la distribución porcentual de la variable Mode
print(df["Mode"].value_counts(normalize=True) * 100)

# 2. Particionado estratificado considerando 'Mode' como variable objetivo
X = df.drop(columns=["Mode"])
y = df["Mode"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Mode
1    68.937876
0    31.062124
Name: proportion, dtype: float64
